# MDR-TS v19.3
**Temporal-Station Base Model Benchmark for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR  
**Notebook Type:** Training & Evaluation  
**Last Updated:** Thu Apr 16 2026

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v19.3
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `Temporal/Pipeline/data/splits/derived_8.0/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

## What's New?
- Forked from `v19.2` on the `derived_8.0` feature set
- Replaced the two-branch XGBoost drift check with a five-model benchmark
- Compares **XGBoost**, **CatBoost**, **LightGBM**, **Random Forest**, and **sklearn MLP**
- Leaves the **PyTorch MLP** scaffold in place but commented out of the active benchmark
- Keeps the same weighted train+val to held-out test-station evaluation structure


## 0. Imports

In [ ]:
import math
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

try:
    from xgboost import XGBRegressor
    import xgboost
except ImportError:
    XGBRegressor = None
    xgboost = None

try:
    from catboost import CatBoostRegressor
    import catboost
except ImportError:
    CatBoostRegressor = None
    catboost = None

try:
    from lightgbm import LGBMRegressor
    import lightgbm
except ImportError:
    LGBMRegressor = None
    lightgbm = None

import torch

def find_models_temporal_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for cand in candidates:
        if (cand / "Models/Temporal/Utils/dashboard.py").exists():
            return cand / "Models/Temporal"
        if (cand / "Utils/dashboard.py").exists():
            return cand
    raise FileNotFoundError("Could not locate Models/Temporal for Utils imports")

TEMPORAL_MODELS_ROOT = find_models_temporal_root()
if str(TEMPORAL_MODELS_ROOT) not in sys.path:
    sys.path.append(str(TEMPORAL_MODELS_ROOT))

from Utils.dashboard import metrics_dashboard

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))


In [ ]:
SEED = 42
DEEP_SEARCH = 40

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

def _version_or_missing(module_obj, name):
    if module_obj is None:
        return f"{name} not installed"
    return getattr(module_obj, "__version__", "unknown")

def print_env_info():
    print("Environment information:")
    print(f"  Python version:   {os.sys.version.split()[0]}")
    print(f"  NumPy version:    {np.__version__}")
    print(f"  Pandas version:   {pd.__version__}")
    print(f"  XGBoost version:  {_version_or_missing(xgboost, 'xgboost')}")
    print(f"  CatBoost version: {_version_or_missing(catboost, 'catboost')}")
    print(f"  LightGBM version: {_version_or_missing(lightgbm, 'lightgbm')}")

    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")


In [ ]:
VERSION = "v19"
SUBVERSION = "v19.3"
RUN_NAME = "mdr_ts_v19_3"

def find_project_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for cand in candidates:
        if (cand / "Temporal/Pipeline/data").exists() and (cand / "Models/Temporal").exists():
            return cand
    raise FileNotFoundError("Could not locate repo root containing Temporal/Pipeline/data")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "Temporal/Pipeline/data"
SPLIT_ROOT = DATA_ROOT / "splits"
OUTPUT_ROOT = PROJECT_ROOT / "Models/Temporal" / VERSION / SUBVERSION

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:", DATA_ROOT.exists())
print("  splits exists:", SPLIT_ROOT.exists())
print("  output exists:", OUTPUT_ROOT.exists())


In [ ]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_8.0/train.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_8.0/val.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_8.0/test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

In [ ]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

In [ ]:
TARGET_COL = "soil_moisture_5cm"

KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS = [
    'SMAP_sm_pm_interp_ema02',
    'V_rollmin_LST_modis_kobs30',
    'D_sin_DOY', 'G_rain_sum_3d',
    'V_ema_G_API_kobs7',
    'V_rollmin_G_API_kobs30',
    'G_rain_sum_7d',
    'C_lag_LST_modis_kobs30',
    'C_lag_G_API_kobs1',
    'V_ema_G_API_kobs14',
    'V_rollmean_G_API_kobs14',
    'G_API', 'G_DSLR',
    'SMAP_ampm_diff_interp',
    'V_rollmax_G_API_kobs30',
    'V_ema_G_API_kobs30',
    'V_rollmean_s2_b11_kobs7',
    'V_ema_LST_modis_kobs7',
    'V_rollmean_G_API_kobs7',
    'C_lag_s2_b11_kobs30',
    'A_d_E_SAR_diff_kobs14',
    'C_lag_LST_modis_kobs6',
    'A_d_LST_modis_kobs14',
    'A_d_SMAP_sm_interp_kobs14',
    'V_rollstd_SMAP_sm_interp_kobs30',
    'SMAP_sm_interp_grad7',
    'year_frac', 'sin_year', 'cos_year',
    'API_x_year', 'SMAP_x_year',
    'slope', 'elev', 'K_slope_sin',
    'K_slope_cos', 'K_aspect_cos',
    'J_clay_wfrac_b0', 'J_sand_wfrac_b0'
    ]

expected = set(KEEP_META_COLS + FEATURE_COLS + [TARGET_COL])
missing_train = sorted(list(expected - set(train_df.columns)))
missing_val   = sorted(list(expected - set(val_df.columns)))
missing_test  = sorted(list(expected - set(test_df.columns)))

if missing_train or missing_val or missing_test:
    raise ValueError(
        f"Missing columns:\n"
        f"  train: {missing_train}\n"
        f"  val:   {missing_val}\n"
        f"  test:  {missing_test}"
    )

print("Columns locked")
print("  Features:", len(FEATURE_COLS))
print("  Target:  ", TARGET_COL)

In [ ]:
corr = train_df[FEATURE_COLS].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# find pairs > 0.995 correlation
high_corr = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.995
]

print("Highly correlated pairs:")
for a, b, c in high_corr:
    print(f"{a} <-> {b} : {c:.5f}")

In [ ]:
def get_metrics_dict(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    err = y_true - y_pred
    ae = np.abs(err)

    r2 = float(r2_score(y_true, y_pred))
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(root_mean_squared_error(y_true, y_pred))

    bias = float(np.mean(err))
    ubrmse = float(np.std(err))

    q_err = np.quantile(err, [0.05, 0.25, 0.50, 0.75, 0.95])

    return {
        f"{prefix}n": int(len(y_true)),
        f"{prefix}r2": r2,
        f"{prefix}mae": mae,
        f"{prefix}rmse": rmse,
        f"{prefix}ubrmse": ubrmse,
        f"{prefix}bias": bias,
        f"{prefix}med_ae": float(np.median(ae)),
        f"{prefix}p90_ae": float(np.quantile(ae, 0.90)),
        f"{prefix}q05_err": float(q_err[0]),
        f"{prefix}q50_err": float(q_err[2]),
        f"{prefix}q95_err": float(q_err[4]),
    }

def neat_print(metrics):
    print(f"{'METRIC':<15} | {'VALUE':<10}")
    print("-" * 28)
    for k, v in metrics.items():
        val_str = f"{v:,}" if isinstance(v, int) else f"{v:+.5f}"
        print(f"{k:<15} | {val_str:<10}")

### Split Strategy
- **Training set:**  
  Two stations, early time period  
- **Validation set:**  
  Same stations as training, held-out **future dates** (temporal holdout)
- **Test set:**  
  One completely unseen station (station-level holdout)

### Motivation
- Validation evaluates **temporal generalization** on known stations
- Test evaluates **spatial generalization** to an unseen station
- This avoids spatial leakage while preserving sufficient training data

In [ ]:
print("=== SPLIT SUMMARY ===")

def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== LEAKAGE CHECK ===")
print("train ∩ test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   ∩ test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- split locked --")


In [ ]:
trainval_df_d = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

trainval_df_d["date"] = pd.to_datetime(trainval_df_d["date"], errors="coerce")
trainval_df_d["year"] = trainval_df_d["date"].dt.year.astype(float)

max_year = trainval_df_d["year"].max()
beta = 0.2

w_trainval = np.exp(beta * (trainval_df_d["year"] - max_year))
w_trainval = w_trainval / w_trainval.mean()

In [ ]:
trainval_df_d = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

X_trainval_d = trainval_df_d[FEATURE_COLS].copy()
y_trainval_d = trainval_df_d[TARGET_COL].copy()

X_test_d = test_df[FEATURE_COLS].copy()
y_test_d = test_df[TARGET_COL].copy()

print("\nDRIFT matrices:")
print("  X_trainval_d:", X_trainval_d.shape)
print("  X_test_d:    ", X_test_d.shape)

### Base Model Benchmark

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

if str(OUTPUT_ROOT) not in sys.path:
    sys.path.append(str(OUTPUT_ROOT))

from torch_mlp_regressor import TorchMLPRegressor

MODEL_DISPLAY = {
    "xgboost": "XGBoost",
    "catboost": "CatBoost",
    "lightgbm": "LightGBM",
    "random_forest": "Random Forest",
    "sklearn_mlp": "Sklearn MLP",
    # "torch_mlp": "PyTorch MLP",
}

XGB_PARAMS_BASE = dict(
    objective="reg:pseudohubererror",
    random_state=SEED,
    n_jobs=-1,
    subsample=0.9,
    colsample_bytree=0.8,
    max_depth=8,
    min_child_weight=2,
    n_estimators=5500,
    learning_rate=0.04,
    reg_lambda=1.5,
    reg_alpha=0.03,
    gamma=0.0,
)

CATBOOST_PARAMS_BASE = dict(
    loss_function="MAE",
    eval_metric="RMSE",
    random_seed=SEED,
    iterations=2500,
    learning_rate=0.04,
    depth=8,
    l2_leaf_reg=3.0,
    subsample=0.9,
    verbose=False,
)

LGBM_PARAMS_BASE = dict(
    objective="regression_l1",
    random_state=SEED,
    n_estimators=2500,
    learning_rate=0.04,
    num_leaves=63,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_lambda=1.5,
    reg_alpha=0.03,
    min_child_samples=20,
    n_jobs=-1,
    verbosity=-1,
)

RF_PARAMS_BASE = dict(
    random_state=SEED,
    n_estimators=1000,
    max_depth=20,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
)

SKLEARN_MLP_BASE = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-3,
    batch_size=256,
    learning_rate="adaptive",
    learning_rate_init=1e-3,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=30,
    random_state=SEED,
)

SKLEARN_NN_PARAMS_BASE = dict(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("mlp", SKLEARN_MLP_BASE),
    ]
)

TORCH_NN_PARAMS_BASE = dict(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "mlp",
            TorchMLPRegressor(
                hidden_dims=(128, 64),
                lr=1e-3,
                weight_decay=1e-4,
                epochs=500,
                batch_size=256,
                patience=40,
                dropout=0.15,
                random_state=SEED,
            ),
        ),
    ]
)

required_libs = []
if XGBRegressor is None:
    required_libs.append("xgboost")
if CatBoostRegressor is None:
    required_libs.append("catboost")
if LGBMRegressor is None:
    required_libs.append("lightgbm")

if required_libs:
    raise ImportError(
        "v19.3 requires the following packages for the benchmark: "
        + ", ".join(required_libs)
    )

BENCHMARK_SPECS = [
    ("xgboost", XGBRegressor, XGB_PARAMS_BASE),
    ("catboost", CatBoostRegressor, CATBOOST_PARAMS_BASE),
    ("lightgbm", LGBMRegressor, LGBM_PARAMS_BASE),
    ("random_forest", RandomForestRegressor, RF_PARAMS_BASE),
    ("sklearn_mlp", Pipeline, SKLEARN_NN_PARAMS_BASE),
    # ("torch_mlp", Pipeline, TORCH_NN_PARAMS_BASE),
]


In [ ]:
benchmark_rows = []
benchmark_models = {}
benchmark_preds = {}

w = w_trainval.values if hasattr(w_trainval, "values") else w_trainval

for model_key, model_cls, params in BENCHMARK_SPECS:
    print(f"\n===== TRAINING {MODEL_DISPLAY[model_key].upper()} =====")
    model = model_cls(**params)

    fit_kwargs = {}
    if model_key in {"xgboost", "catboost", "lightgbm", "random_forest"}:
        fit_kwargs["sample_weight"] = w
    elif model_key == "sklearn_mlp":
        fit_kwargs["mlp__sample_weight"] = w
    if model_key == "xgboost":
        fit_kwargs["verbose"] = 0

    model.fit(X_trainval_d, y_trainval_d, **fit_kwargs)
    pred_test = np.asarray(model.predict(X_test_d)).ravel()
    metrics = get_metrics_dict(y_test_d, pred_test, prefix="")

    benchmark_models[model_key] = model
    benchmark_preds[model_key] = pred_test
    benchmark_rows.append({
        "model_key": model_key,
        "model": MODEL_DISPLAY[model_key],
        **metrics,
    })

benchmark_df = (
    pd.DataFrame(benchmark_rows)
    .sort_values(["r2", "rmse", "mae"], ascending=[False, True, True])
    .reset_index(drop=True)
)

best_model_key = benchmark_df.loc[0, "model_key"]
best_model_name = benchmark_df.loc[0, "model"]
best_model = benchmark_models[best_model_key]
best_pred = benchmark_preds[best_model_key]


In [ ]:
benchmark_view = benchmark_df[
    ["model", "n", "r2", "mae", "rmse", "ubrmse", "bias", "med_ae", "p90_ae"]
].copy()

print("===== BASE MODEL BENCHMARK =====")
display(benchmark_view)


In [ ]:
print(f"Best base model: {best_model_name}")
metrics_dashboard(
    y_test_d,
    best_pred,
    name=f"Best Base Model: {best_model_name}",
    return_dict=False,
)

print("\n===== BEST MODEL METRICS =====")
neat_print(get_metrics_dict(y_test_d, best_pred, prefix="best_"))


### Metric Comparison

In [ ]:
plot_metrics = ["r2", "mae", "rmse", "ubrmse", "bias"]
fig, axes = plt.subplots(1, len(plot_metrics), figsize=(22, 4))

for ax, metric in zip(axes, plot_metrics):
    vals = benchmark_df[metric].values
    ax.bar(benchmark_df["model"], vals)
    ax.set_title(metric.upper())
    ax.tick_params(axis="x", rotation=30)
    if metric == "bias":
        ax.axhline(0, color="black", linewidth=1, alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
def get_model_importance(model, feature_cols):
    if hasattr(model, "feature_importances_"):
        imp = np.asarray(model.feature_importances_, dtype=float)
    elif hasattr(model, "get_feature_importance"):
        imp = np.asarray(model.get_feature_importance(), dtype=float)
    else:
        return None

    imp = np.nan_to_num(imp, nan=0.0, posinf=0.0, neginf=0.0)
    total = imp.sum()
    if total > 0:
        imp = imp / total
    return pd.Series(imp, index=feature_cols)

importance_series = {}
for k, m in benchmark_models.items():
    imp = get_model_importance(m, FEATURE_COLS)
    if imp is not None:
        importance_series[MODEL_DISPLAY[k]] = imp

importance_df = pd.DataFrame(importance_series)

importance_df["consensus_mean"] = importance_df.mean(axis=1)
TOP_K = 30
importance_top = importance_df.sort_values("consensus_mean", ascending=False).head(TOP_K)
importance_plot = importance_top.sort_values("consensus_mean", ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
y = np.arange(len(importance_plot))
bar_h = 0.18

for i, model_name in enumerate(importance_series.keys()):
    ax.barh(
        y + (i - (len(importance_series) - 1) / 2) * bar_h,
        importance_plot[model_name].values,
        height=bar_h,
        label=model_name,
    )

ax.set_yticks(y)
ax.set_yticklabels(importance_plot.index)
ax.set_xlabel("Normalized Importance")
ax.set_title(f"Top {TOP_K} Feature Importances Across Base Models (tree-based only)")
ax.grid(axis="x", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

display(importance_top.reset_index().rename(columns={"index": "feature"}))


In [ ]:
model_keys = [x[0] for x in BENCHMARK_SPECS]
n_models = len(model_keys)
ncols = 2
nrows = math.ceil(n_models / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4.5 * nrows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

for ax, model_key in zip(axes, model_keys):
    pred = benchmark_preds[model_key]
    resid = y_test_d - pred
    ax.scatter(y_test_d, resid, s=8, alpha=0.55)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(MODEL_DISPLAY[model_key])
    ax.set_xlabel("True Soil Moisture")
    ax.set_ylabel("Residual (true - pred)")

for ax in axes[n_models:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
diag_2023_rows = []
diag_dates = pd.to_datetime(test_df["date"], errors="coerce")
mask_2023 = diag_dates.dt.year == 2023

for model_key in [x[0] for x in BENCHMARK_SPECS]:
    pred = benchmark_preds[model_key]
    if mask_2023.sum() >= 2:
        y_2023 = y_test_d[mask_2023]
        pred_2023 = pred[mask_2023]
        slope, intercept = np.polyfit(pred_2023, y_2023, 1)
        diag_2023_rows.append({
            "model": MODEL_DISPLAY[model_key],
            "n_2023": int(mask_2023.sum()),
            "r2_2023": float(r2_score(y_2023, pred_2023)),
            "slope_2023": float(slope),
            "intercept_2023": float(intercept),
        })

diag_2023_df = pd.DataFrame(diag_2023_rows)
print("===== 2023 SLICE DIAGNOSTICS =====")
display(diag_2023_df)


In [ ]:
model_keys = [x[0] for x in BENCHMARK_SPECS]
n_models = len(model_keys)
ncols = 2
nrows = math.ceil(n_models / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4.5 * nrows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

line_min = float(np.min(y_test_d))
line_max = float(np.max(y_test_d))

for ax, model_key in zip(axes, model_keys):
    pred = benchmark_preds[model_key]
    ax.scatter(y_test_d, pred, s=8, alpha=0.55)
    ax.plot([line_min, line_max], [line_min, line_max], linestyle="--", linewidth=1)
    ax.set_title(MODEL_DISPLAY[model_key])
    ax.set_xlabel("True Soil Moisture")
    ax.set_ylabel("Predicted Soil Moisture")

for ax in axes[n_models:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


### Artifact Export

In [ ]:
benchmark_pred_df = test_df[["station_id", "date"]].copy()
benchmark_pred_df["y_true"] = y_test_d

for model_key in [x[0] for x in BENCHMARK_SPECS]:
    benchmark_pred_df[f"pred_{model_key}"] = benchmark_preds[model_key]

benchmark_metrics_path = Path(OUTPUT_ROOT) / "base_model_benchmark_metrics.csv"
benchmark_preds_path = Path(OUTPUT_ROOT) / "base_model_benchmark_predictions.csv"
benchmark_2023_path = Path(OUTPUT_ROOT) / "base_model_benchmark_2023_slice.csv"
benchmark_importance_path = Path(OUTPUT_ROOT) / "base_model_feature_importance_top30.csv"

benchmark_df.to_csv(benchmark_metrics_path, index=False)
benchmark_pred_df.to_csv(benchmark_preds_path, index=False)
diag_2023_df.to_csv(benchmark_2023_path, index=False)
importance_top.reset_index().rename(columns={"index": "feature"}).to_csv(
    benchmark_importance_path,
    index=False,
)


In [ ]:
print("Saved artifacts:")
print(" ", benchmark_metrics_path)
print(" ", benchmark_preds_path)
print(" ", benchmark_2023_path)
print(" ", benchmark_importance_path)
print(f"\nBenchmark complete. Best base model: {best_model_name}")
